# NB04 · Comparación YOLO11 multisemilla

**Entorno obligatorio:** GPU NVIDIA  
**Recomendado:** L4  
**Tiempo estimado:** L4, 20–35 min; T4, 30–50 min  
**CPU:** no recomendado

Se comparan tres tamaños del clasificador YOLO11 con tres semillas:

- `yolo11n-cls.pt`;
- `yolo11s-cls.pt`;
- `yolo11m-cls.pt`;
- semillas 2026, 2027 y 2028.

El notebook reutiliza la corrida `YOLO11n · semilla 2026` de NB03. Las demás corridas se guardan inmediatamente en Drive. Si Colab se desconecta, vuelva a ejecutar todo: las corridas terminadas se detectarán y no se repetirán.

> La arquitectura ganadora se determina con validación, no con prueba. El conjunto de prueba se usa para estimar la generalización final.


In [ ]:
%pip install -q -U ultralytics scikit-learn seaborn


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import gc
import json
import platform
import random
import shutil
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import torch
import ultralytics
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from ultralytics import YOLO

if not torch.cuda.is_available():
    raise RuntimeError(
        'No se detectó GPU. Active T4 o L4 en Entorno de ejecución > '
        'Cambiar tipo de entorno de ejecución.'
    )

GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {GPU_NAME} ({GPU_MEMORY_GB:.1f} GB)')
print('PyTorch:', torch.__version__)
print('Ultralytics:', ultralytics.__version__)


In [ ]:
PROJECT_ROOT = Path('/content/drive/MyDrive/PALMA_AFRICANA_YOLO')
DRIVE_DATA = PROJECT_ROOT / '02_DATOS_PROCESADOS' / 'classification_5class_v1'
MODELS_ROOT = PROJECT_ROOT / '05_MODELOS'
RESULTS_ROOT = PROJECT_ROOT / '06_RESULTADOS'
NB04_RESULTS = RESULTS_ROOT / 'NB04_COMPARACION_YOLO11_MULTISEED'
NB03_MODEL = MODELS_ROOT / 'YOLO11n_cls_seed2026' / 'best.pt'
NB03_METRICS = RESULTS_ROOT / 'NB03_YOLO11n_cls_seed2026' / 'metricas_test.json'

LOCAL_DATA = Path('/content/PALMA_CLS')
LOCAL_RUNS = Path('/content/palma_nb04_runs')
CLASSES = ['damaged', 'empty', 'overripe', 'ripe', 'unripe']
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

EXPERIMENTS = [
    {'model_key': 'yolo11n', 'weights': 'yolo11n-cls.pt', 'batch': 16, 'seed': seed}
    for seed in [2026, 2027, 2028]
] + [
    {'model_key': 'yolo11s', 'weights': 'yolo11s-cls.pt', 'batch': 16, 'seed': seed}
    for seed in [2026, 2027, 2028]
] + [
    {'model_key': 'yolo11m', 'weights': 'yolo11m-cls.pt', 'batch': 8, 'seed': seed}
    for seed in [2026, 2027, 2028]
]

for path in [NB04_RESULTS, MODELS_ROOT, LOCAL_RUNS]:
    path.mkdir(parents=True, exist_ok=True)
if not DRIVE_DATA.exists():
    raise FileNotFoundError('No se encontró el dataset procesado. Ejecute NB02.')

display(pd.DataFrame(EXPERIMENTS))


In [ ]:
# Copia local para evitar que Drive limite la velocidad durante las nueve corridas.
if LOCAL_DATA.exists():
    shutil.rmtree(LOCAL_DATA)
shutil.copytree(DRIVE_DATA, LOCAL_DATA)

distribution_rows = []
for split in ['train', 'val', 'test']:
    for label in CLASSES:
        folder = LOCAL_DATA / split / label
        n = sum(1 for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
        distribution_rows.append({'split': split, 'class': label, 'n': n})
distribution = pd.DataFrame(distribution_rows)
display(distribution.pivot(index='class', columns='split', values='n'))

assert distribution.query("split == 'train'")['n'].sum() == 312
assert distribution.query("split == 'val'")['n'].sum() == 70
assert distribution.query("split == 'test'")['n'].sum() == 70
print('Datos locales listos.')


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def split_paths(split):
    return sorted(
        p for p in (LOCAL_DATA / split).glob('*/*')
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    )


def evaluate_model(model, split, imgsz, batch, run_name, output_dir):
    paths = split_paths(split)
    start = time.perf_counter()
    outputs = model.predict(
        source=[str(p) for p in paths],
        imgsz=imgsz,
        batch=batch,
        device=0,
        verbose=False,
    )
    seconds = time.perf_counter() - start

    rows = []
    for path, result in zip(paths, outputs):
        probs = result.probs.data.detach().cpu().numpy().astype(float)
        names = {int(k): v for k, v in result.names.items()}
        pred_id = int(np.argmax(probs))
        row = {
            'run_name': run_name,
            'split': split,
            'image': path.name,
            'relative_path': str(path.relative_to(LOCAL_DATA)),
            'true_class': path.parent.name,
            'pred_class': names[pred_id],
            'confidence': float(probs[pred_id]),
            'correct': bool(path.parent.name == names[pred_id]),
        }
        for class_id, probability in enumerate(probs):
            row[f'prob_{names[class_id]}'] = float(probability)
        rows.append(row)

    frame = pd.DataFrame(rows)
    y_true, y_pred = frame['true_class'], frame['pred_class']
    report = classification_report(
        y_true, y_pred, labels=CLASSES, target_names=CLASSES,
        output_dict=True, zero_division=0,
    )
    metrics = {
        f'{split}_n': int(len(frame)),
        f'{split}_accuracy': float(accuracy_score(y_true, y_pred)),
        f'{split}_balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        f'{split}_precision_macro': float(precision_score(y_true, y_pred, labels=CLASSES, average='macro', zero_division=0)),
        f'{split}_recall_macro': float(recall_score(y_true, y_pred, labels=CLASSES, average='macro', zero_division=0)),
        f'{split}_f1_macro': float(f1_score(y_true, y_pred, labels=CLASSES, average='macro', zero_division=0)),
        f'{split}_f1_weighted': float(f1_score(y_true, y_pred, labels=CLASSES, average='weighted', zero_division=0)),
        f'{split}_inference_ms_image': float(seconds * 1000 / len(frame)),
    }
    for label in CLASSES:
        metrics[f'{split}_f1_{label}'] = float(report[label]['f1-score'])

    frame.to_csv(output_dir / f'predicciones_{split}.csv', index=False)
    pd.DataFrame(report).T.to_csv(output_dir / f'reporte_{split}.csv')
    pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=CLASSES),
        index=CLASSES, columns=CLASSES,
    ).to_csv(output_dir / f'matriz_confusion_{split}.csv')
    return metrics


In [ ]:
EPOCHS = 60
IMGSZ = 384
PATIENCE = 15

def run_experiment(exp):
    model_key = exp['model_key']
    seed = exp['seed']
    batch = exp['batch']
    run_name = f'{model_key}_cls_seed{seed}'
    run_results = NB04_RESULTS / run_name
    run_model_dir = MODELS_ROOT / run_name
    summary_path = run_results / 'resumen.json'
    saved_best = run_model_dir / 'best.pt'
    run_results.mkdir(parents=True, exist_ok=True)
    run_model_dir.mkdir(parents=True, exist_ok=True)

    if summary_path.exists() and saved_best.exists():
        print(f'REUTILIZADA: {run_name}')
        return json.loads(summary_path.read_text(encoding='utf-8'))

    set_seed(seed)
    reused_nb03 = model_key == 'yolo11n' and seed == 2026 and NB03_MODEL.exists()
    if reused_nb03:
        print(f'REUTILIZANDO NB03: {run_name}')
        best_path = NB03_MODEL
        training_minutes = None
        if NB03_METRICS.exists():
            nb03 = json.loads(NB03_METRICS.read_text(encoding='utf-8'))
            training_minutes = nb03.get('training_minutes')
    else:
        print(f'ENTRENANDO: {run_name}')
        local_run = LOCAL_RUNS / run_name
        start = time.time()
        trainer = YOLO(exp['weights'])
        trainer.train(
            data=str(LOCAL_DATA), epochs=EPOCHS, imgsz=IMGSZ,
            batch=batch, patience=PATIENCE, device=0, workers=2,
            seed=seed, deterministic=True, pretrained=True,
            optimizer='auto', amp=True, cache=True,
            project=str(LOCAL_RUNS), name=run_name, exist_ok=True,
            plots=True, verbose=False,
        )
        training_minutes = (time.time() - start) / 60
        best_path = local_run / 'weights' / 'best.pt'
        if not best_path.exists():
            raise FileNotFoundError(f'No se generó {best_path}')
        for item in ['results.csv', 'results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png', 'args.yaml']:
            source = local_run / item
            if source.exists():
                shutil.copy2(source, run_model_dir / item)

    if best_path.resolve() != saved_best.resolve():
        shutil.copy2(best_path, saved_best)
    model_size_mb = saved_best.stat().st_size / 1024**2
    best_model = YOLO(str(saved_best))
    val_metrics = evaluate_model(best_model, 'val', IMGSZ, batch, run_name, run_results)
    test_metrics = evaluate_model(best_model, 'test', IMGSZ, batch, run_name, run_results)

    summary = {
        'run_name': run_name,
        'model_key': model_key,
        'weights': exp['weights'],
        'seed': seed,
        'batch': batch,
        'imgsz': IMGSZ,
        'epochs_max': EPOCHS,
        'patience': PATIENCE,
        'gpu': GPU_NAME,
        'training_minutes': training_minutes,
        'model_size_mb': float(model_size_mb),
        'reused_nb03': reused_nb03,
        **val_metrics,
        **test_metrics,
    }
    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f"FINALIZADA: {run_name} | val F1={summary['val_f1_macro']:.4f} | test F1={summary['test_f1_macro']:.4f}")

    del best_model
    torch.cuda.empty_cache()
    gc.collect()
    return summary


In [ ]:
# Ejecuta o reanuda las nueve corridas. El resumen se actualiza después de cada una.
all_summaries = []
for position, exp in enumerate(EXPERIMENTS, start=1):
    print(f'\n=== Corrida {position}/{len(EXPERIMENTS)} ===')
    summary = run_experiment(exp)
    all_summaries.append(summary)
    pd.DataFrame(all_summaries).to_csv(NB04_RESULTS / 'resultados_parciales.csv', index=False)

runs = pd.DataFrame(all_summaries).sort_values(['model_key', 'seed']).reset_index(drop=True)
runs.to_csv(NB04_RESULTS / 'resultados_por_corrida.csv', index=False)
display(runs[[
    'model_key', 'seed', 'val_accuracy', 'val_f1_macro',
    'test_accuracy', 'test_f1_macro', 'training_minutes',
    'model_size_mb', 'test_inference_ms_image'
]].round(4))


In [ ]:
# Resumen por arquitectura. La clasificación se ordena por F1 macro de VALIDACIÓN.
aggregate = runs.groupby('model_key').agg(
    val_accuracy_mean=('val_accuracy', 'mean'),
    val_accuracy_std=('val_accuracy', 'std'),
    val_f1_macro_mean=('val_f1_macro', 'mean'),
    val_f1_macro_std=('val_f1_macro', 'std'),
    test_accuracy_mean=('test_accuracy', 'mean'),
    test_accuracy_std=('test_accuracy', 'std'),
    test_f1_macro_mean=('test_f1_macro', 'mean'),
    test_f1_macro_std=('test_f1_macro', 'std'),
    training_minutes_mean=('training_minutes', 'mean'),
    model_size_mb_mean=('model_size_mb', 'mean'),
    inference_ms_image_mean=('test_inference_ms_image', 'mean'),
).reset_index()
aggregate = aggregate.sort_values('val_f1_macro_mean', ascending=False).reset_index(drop=True)
aggregate.insert(0, 'validation_rank', np.arange(1, len(aggregate) + 1))
aggregate.to_csv(NB04_RESULTS / 'resumen_por_modelo.csv', index=False)
display(aggregate.round(4))

selected_model = aggregate.iloc[0]['model_key']
selection = {
    'selection_rule': 'Mayor F1 macro promedio en validación; prueba no usada para seleccionar.',
    'selected_model': selected_model,
    'seeds': [2026, 2027, 2028],
}
(NB04_RESULTS / 'seleccion_modelo.json').write_text(
    json.dumps(selection, ensure_ascii=False, indent=2), encoding='utf-8'
)
print('Arquitectura seleccionada por validación:', selected_model)


In [ ]:
# Figuras comparativas para informe y libro.
sns.set_theme(style='whitegrid', context='notebook')
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

sns.boxplot(data=runs, x='model_key', y='val_f1_macro', color='#9ecae1', ax=axes[0])
sns.stripplot(data=runs, x='model_key', y='val_f1_macro', color='#08519c', size=7, ax=axes[0])
axes[0].set_title('F1 macro en validación · tres semillas')
axes[0].set_xlabel('Arquitectura')
axes[0].set_ylabel('F1 macro')
axes[0].set_ylim(0, 1.03)

sns.boxplot(data=runs, x='model_key', y='test_f1_macro', color='#a1d99b', ax=axes[1])
sns.stripplot(data=runs, x='model_key', y='test_f1_macro', color='#006d2c', size=7, ax=axes[1])
axes[1].set_title('F1 macro en prueba · estimación final')
axes[1].set_xlabel('Arquitectura')
axes[1].set_ylabel('F1 macro')
axes[1].set_ylim(0, 1.03)

comparison_path = NB04_RESULTS / 'comparacion_f1_macro_multisemilla.png'
fig.savefig(comparison_path, dpi=220, bbox_inches='tight')
plt.show()

class_columns = [f'test_f1_{label}' for label in CLASSES]
class_means = runs.groupby('model_key')[class_columns].mean()
class_means.columns = CLASSES
class_means.to_csv(NB04_RESULTS / 'f1_test_por_clase_promedio.csv')

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.heatmap(class_means, annot=True, fmt='.3f', cmap='YlGnBu', vmin=0, vmax=1, ax=ax)
ax.set_title('F1 promedio por clase en prueba · tres semillas')
ax.set_xlabel('Clase')
ax.set_ylabel('Arquitectura')
heatmap_path = NB04_RESULTS / 'f1_test_por_clase_heatmap.png'
fig.savefig(heatmap_path, dpi=220, bbox_inches='tight')
plt.show()


In [ ]:
environment = {
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'ultralytics': ultralytics.__version__,
    'scikit_learn': sklearn.__version__,
    'gpu': GPU_NAME,
    'gpu_memory_gb': round(GPU_MEMORY_GB, 2),
    'models': [exp['weights'] for exp in EXPERIMENTS],
    'seeds': [2026, 2027, 2028],
}
(NB04_RESULTS / 'entorno_ejecucion.json').write_text(
    json.dumps(environment, ensure_ascii=False, indent=2), encoding='utf-8'
)

print('NB04 FINALIZADO CORRECTAMENTE')
print('Arquitectura seleccionada por validación:', selected_model)
print('Resultados:', NB04_RESULTS)


## Interpretación

1. Compare primero el **F1 macro promedio de validación** y su desviación estándar.
2. Use el resultado de prueba para informar la generalización, no para cambiar hiperparámetros.
3. Examine el F1 de `damaged` y `empty`: sus muestras son muy escasas y un solo error produce grandes cambios.
4. Un modelo mayor no necesariamente será mejor; también deben considerarse tamaño, tiempo e inferencia.

Cuando aparezca `NB04 FINALIZADO CORRECTAMENTE`, informe el resultado para revisar la arquitectura seleccionada y decidir si NB05 debe tratar el desbalance de clases.
